In [1]:
from pathlib import Path
import pandas as pd
import re

# Define project folders
text_folder = Path("../data/extracted_text")
processed_folder = Path("../data/processed")
keyword_file = Path("../data/keyword_dictionary.csv")
company_file = Path("../data/company_master_list.csv")

# Check extracted text files
text_files = sorted(text_folder.glob("*.txt"))

print(f"Extracted text files found: {len(text_files)}")

for file in text_files:
    print(file.name)

Extracted text files found: 20
001_barclays_plc_annual_report_2025.txt
002_hsbc_holdings_plc_annual_report_2025.txt
003_lloyds_banking_group_plc_annual_report_2025.txt
004_natwest_group_plc_annual_report_2025.txt
005_standard_chartered_plc_annual_report_2025.txt
006_aviva_plc_annual_report_2025.txt
007_legal_and_general_group_plc_annual_report_2025.txt
008_m_and_g_plc_annual_report_2025.txt
009_admiral_group_plc_annual_report_2025.txt
010_aj_bell_plc_annual_report_2025.txt
011_ig_group_holdings_plc_annual_report_2025.txt
012_cmc_markets_plc_annual_report_2025.txt
013_integrafin_holdings_plc_annual_report_2025.txt
014_wise_plc_annual_report_2025.txt
015_cab_payments_holdings_plc_annual_report_2025.txt
016_funding_circle_holdings_plc_annual_report_2025.txt
017_boku_inc_annual_report_2025.txt
018_paypoint_plc_annual_report_2025.txt
019_london_stock_exchange_group_plc_annual_report_2025.txt
020_experian_plc_annual_report_2025.txt


In [2]:
keyword_data = [
    
    # AI and related technologies
    ["AI and related technologies", "artificial intelligence", "artificial intelligence", "Direct reference to AI"],
    ["AI and related technologies", "AI", "AI", "Direct acronym reference to artificial intelligence"],
    ["AI and related technologies", "machine learning", "machine learning", "Specific AI-related technique"],
    ["AI and related technologies", "algorithm", "algorithm", "Algorithmic systems or decision processes"],
    ["AI and related technologies", "automation", "automation", "Automated processes linked to digital operations"],
    ["AI and related technologies", "generative AI", "generative AI", "Emerging AI technology"],
    ["AI and related technologies", "large language model", "large language model", "Specific generative AI technology"],
    ["AI and related technologies", "LLM", "LLM", "Acronym for large language model"],

    # Data and analytics
    ["Data and analytics", "data analytics", "data analytics", "Evidence of data-driven operations"],
    ["Data and analytics", "advanced analytics", "advanced analytics", "Evidence of analytical capability"],
    ["Data and analytics", "predictive analytics", "predictive analytics", "Predictive decision-support capability"],
    ["Data and analytics", "data science", "data science", "Technical data capability"],
    ["Data and analytics", "digital transformation", "digital transformation", "Wider digital change context"],

    # Governance and accountability
    ["Governance and accountability", "governance", "governance", "General governance reference"],
    ["Governance and accountability", "accountability", "accountability", "Accountability in decision-making"],
    ["Governance and accountability", "oversight", "oversight", "Oversight of technology or risk"],
    ["Governance and accountability", "responsible AI", "responsible AI", "Explicit responsible AI governance"],
    ["Governance and accountability", "ethical AI", "ethical AI", "Explicit AI ethics disclosure"],
    ["Governance and accountability", "AI ethics", "AI ethics", "Explicit AI ethics disclosure"],
    ["Governance and accountability", "model risk", "model risk", "Risk management of models"],

    # Transparency and explainability
    ["Transparency and explainability", "transparency", "transparency", "Transparency disclosure"],
    ["Transparency and explainability", "explainability", "explainability", "Explainability of systems or models"],
    ["Transparency and explainability", "explainable AI", "explainable AI", "Specific explainable AI reference"],
    ["Transparency and explainability", "interpretability", "interpretability", "Model interpretability reference"],

    # Human oversight and decision-making
    ["Human oversight", "human oversight", "human oversight", "Human control over automated systems"],
    ["Human oversight", "human review", "human review", "Manual review of decisions"],
    ["Human oversight", "human-in-the-loop", "human-in-the-loop", "Human involvement in AI systems"],
    ["Human oversight", "automated decision-making", "automated decision-making", "Automated decision processes"],
    ["Human oversight", "decision support", "decision support", "Technology supporting human decisions"],

    # Fairness, bias and ethics
    ["Fairness, bias and ethics", "bias", "bias", "Potential bias in models or decisions"],
    ["Fairness, bias and ethics", "fairness", "fairness", "Fairness in decisions or systems"],
    ["Fairness, bias and ethics", "discrimination", "discrimination", "Potential unfair treatment"],
    ["Fairness, bias and ethics", "ethics", "ethics", "General ethics reference"],

    # Risk, regulation and protection
    ["Risk and regulation", "risk management", "risk management", "Technology risk management"],
    ["Risk and regulation", "regulatory compliance", "regulatory compliance", "Regulatory compliance disclosure"],
    ["Risk and regulation", "data protection", "data protection", "Protection of personal or customer data"],
    ["Risk and regulation", "privacy", "privacy", "Privacy-related disclosure"],
    ["Risk and regulation", "cybersecurity", "cybersecurity", "Technology and cyber risk disclosure"]
]

keyword_dictionary = pd.DataFrame(
    keyword_data,
    columns=[
        "category",
        "keyword",
        "search_variant",
        "reason_for_inclusion"
    ]
)

keyword_dictionary.to_csv(
    keyword_file,
    index=False
)

print(f"Keyword dictionary saved to: {keyword_file}")
print(f"Number of keywords: {len(keyword_dictionary)}")

keyword_dictionary

Keyword dictionary saved to: ..\data\keyword_dictionary.csv
Number of keywords: 38


,category,keyword,search_variant,reason_for_inclusion
0,AI and related technologies,artificial intelligence,artificial intelligence,Direct reference to AI
1,AI and related technologies,AI,AI,Direct acronym reference to artificial intelli...
2,AI and related technologies,machine learning,machine learning,Specific AI-related technique
3,AI and related technologies,algorithm,algorithm,Algorithmic systems or decision processes
4,AI and related technologies,automation,automation,Automated processes linked to digital operations
5,AI and related technologies,generative AI,generative AI,Emerging AI technology
6,AI and related technologies,large language model,large language model,Specific generative AI technology
7,AI and related technologies,LLM,LLM,Acronym for large language model
8,Data and analytics,data analytics,data analytics,Evidence of data-driven operations
9,Data and analytics,advanced analytics,advanced analytics,Evidence of analytical capability


In [3]:
def split_text_into_pages(full_text):
    """
    Splits extracted report text using the page markers created
    in the text extraction notebook.
    """
    
    page_sections = re.split(
        r"===== PAGE (\d+) =====",
        full_text
    )
    
    pages = []
    
    for index in range(1, len(page_sections), 2):
        page_number = int(page_sections[index])
        page_text = page_sections[index + 1]
        pages.append((page_number, page_text))
        
    return pages


def get_context(text, start, end, window=220):
    """
    Returns a short text extract around a matched keyword.
    """
    
    context_start = max(0, start - window)
    context_end = min(len(text), end + window)
    
    context = text[context_start:context_end]
    
    # Clean excess spacing
    context = re.sub(r"\s+", " ", context).strip()
    
    return context


# Short acronyms are matched case-sensitively. Page-level extraction from PDFs
# routinely splits words across column and line breaks, so a case-insensitive
# search finds bare lowercase fragments of ordinary words (for example
# "sustainability" extracted as "sust ai nabilit") and counts them as AI
# disclosures. The letter-boundary lookarounds alone cannot prevent this,
# because the surrounding characters are whitespace rather than letters.

CASE_SENSITIVE_VARIANTS = {"AI", "LLM"}


def build_keyword_pattern(keyword):
    """
    Builds a search pattern.
    Uses stricter boundaries for short acronyms such as AI and LLM.
    """
    
    escaped_keyword = re.escape(keyword)
    
    if keyword in CASE_SENSITIVE_VARIANTS:
        return rf"(?<![A-Za-z]){escaped_keyword}(?![A-Za-z])"
    
    return rf"\b{escaped_keyword}\b"


# Load dictionary and company master file

keyword_dictionary = pd.read_csv(keyword_file)
company_master = pd.read_csv(company_file, dtype={"company_id": str})

all_mentions = []

# Search each extracted report text file

for text_file in text_files:
    
    company_id = text_file.name[:3]
    
    company_row = company_master[
        company_master["company_id"] == company_id
    ]
    
    if not company_row.empty:
        company_name = company_row.iloc[0]["company_name"]
        report_year = company_row.iloc[0]["annual_report_year"]
    else:
        company_name = ""
        report_year = ""
    
    full_text = text_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )
    
    pages = split_text_into_pages(full_text)
    
    for _, keyword_row in keyword_dictionary.iterrows():
        
        category = keyword_row["category"]
        keyword = keyword_row["keyword"]
        search_variant = keyword_row["search_variant"]
        
        pattern = build_keyword_pattern(search_variant)

        # Acronyms are matched case-sensitively; all other terms remain
        # case-insensitive so that sentence-initial capitals still match.
        search_flags = (
            0 if search_variant in CASE_SENSITIVE_VARIANTS else re.IGNORECASE
        )
        
        for page_number, page_text in pages:
            
            for match in re.finditer(
                pattern,
                page_text,
                flags=search_flags
            ):
                
                context = get_context(
                    page_text,
                    match.start(),
                    match.end()
                )
                
                all_mentions.append({
                    "company_id": company_id,
                    "company_name": company_name,
                    "report_year": report_year,
                    "source_file": text_file.name,
                    "category": category,
                    "keyword": keyword,
                    "search_variant": search_variant,
                    "page_number": page_number,
                    "matched_text": match.group(0),
                    "context": context
                })

keyword_mentions = pd.DataFrame(all_mentions)

output_file = processed_folder / "ai_keyword_mentions.csv"

keyword_mentions.to_csv(
    output_file,
    index=False
)

print(f"Total keyword mentions found: {len(keyword_mentions)}")
print(f"Results saved to: {output_file}")

keyword_mentions.head(20)

Total keyword mentions found: 13113
Results saved to: ..\data\processed\ai_keyword_mentions.csv


,company_id,company_name,report_year,source_file,category,keyword,search_variant,page_number,matched_text,context
0,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,42,artificial intelligence,class customer experience; and delivering best...
1,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,62,artificial intelligence,ds (including business or operations; the use ...
2,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,109,artificial intelligence,ustainability reporting standards (including b...
3,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,183,Artificial Intelligence,"al Control, and standardisation of cost report..."
4,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,232,artificial intelligence,ordinary shareholders. legacy businesses; and ...
5,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,233,Artificial Intelligence,ntravention Order Removing Barriers to America...
6,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,233,artificial intelligence,"risks relating to AI technologies.development,..."
7,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,233,Artificial Intelligence,anner or These include the California Transpar...
8,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,233,Artificial Intelligence,-out of new AI tools in Frontier Artificial In...
9,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,238,artificial intelligence,“Evolving landscape with respect to The Group ...


In [4]:
# Validation audit: effect of case-sensitive acronym matching
#
# Re-runs the permissive (case-insensitive) rule for the acronyms only, and
# records every match the stricter rule excludes. This turns the false-positive
# rate into a reportable number for the methodology chapter rather than an
# anecdote, and preserves the excluded extracts as evidence.

false_positive_rows = []

for text_file in text_files:

    company_id = text_file.name[:3]

    company_row = company_master[
        company_master["company_id"] == company_id
    ]

    company_name = (
        company_row.iloc[0]["company_name"]
        if not company_row.empty
        else ""
    )

    full_text = text_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    for page_number, page_text in split_text_into_pages(full_text):

        for variant in sorted(CASE_SENSITIVE_VARIANTS):

            pattern = build_keyword_pattern(variant)

            for match in re.finditer(
                pattern,
                page_text,
                flags=re.IGNORECASE
            ):

                # Kept by the permissive rule, excluded by the strict one
                if match.group(0) != variant:

                    false_positive_rows.append({
                        "company_id": company_id,
                        "company_name": company_name,
                        "search_variant": variant,
                        "page_number": page_number,
                        "matched_text": match.group(0),
                        "context": get_context(
                            page_text,
                            match.start(),
                            match.end(),
                            window=110
                        )
                    })

acronym_false_positives = pd.DataFrame(false_positive_rows)

acronym_false_positives.to_csv(
    processed_folder / "acronym_case_false_positives.csv",
    index=False
)

affected_firms = (
    acronym_false_positives["company_id"].nunique()
    if len(acronym_false_positives) > 0
    else 0
)

print(f"Acronym matches excluded by case-sensitive rule: {len(acronym_false_positives)}")
print(f"Firms affected: {affected_firms} of {len(text_files)}")
print(f"Retained AI-related mentions: {len(keyword_mentions)}")

acronym_false_positives


Acronym matches excluded by case-sensitive rule: 4
Firms affected: 3 of 20
Retained AI-related mentions: 13113


,company_id,company_name,search_variant,page_number,matched_text,context
0,002,HSBC Holdings plc,AI,32,ai,"US and China, including cross-border and AI ar..."
1,002,HSBC Holdings plc,AI,61,ai,sonal customers (IWPB) team corporate customer...
2,005,Standard Chartered plc,AI,30,ai,ly Ch ai 19% ns sets lAs
3,012,CMC Markets plc,AI,32,ai,rs aligns with several United Ourthree sust ai...


In [5]:
# Create output folder for evidence tables

evidence_folder = Path("../outputs/evidence_tables")
evidence_folder.mkdir(parents=True, exist_ok=True)

# Summary 1: keyword mentions by company and category

company_category_summary = (
    keyword_mentions
    .groupby(["company_id", "company_name", "category"])
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique"),
        keywords_found=("keyword", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
)

company_category_summary.to_csv(
    evidence_folder / "company_category_keyword_summary.csv",
    index=False
)

company_category_summary.head(30)

,company_id,company_name,category,mention_count,unique_pages,keywords_found
0,001,Barclays plc,AI and related technologies,231,54,"AI, artificial intelligence, automation, gener..."
1,001,Barclays plc,Data and analytics,10,6,"data analytics, digital transformation"
2,001,Barclays plc,"Fairness, bias and ethics",20,11,"bias, discrimination, ethics, fairness"
3,001,Barclays plc,Governance and accountability,927,491,"accountability, ethical AI, governance, model ..."
4,001,Barclays plc,Human oversight,1,1,human oversight
5,001,Barclays plc,Risk and regulation,358,118,"cybersecurity, data protection, privacy, regul..."
6,001,Barclays plc,Transparency and explainability,41,28,transparency
7,002,HSBC Holdings plc,AI and related technologies,91,33,"AI, artificial intelligence, automation, gener..."
8,002,HSBC Holdings plc,Data and analytics,5,5,"advanced analytics, data analytics, digital tr..."
9,002,HSBC Holdings plc,"Fairness, bias and ethics",17,13,"bias, discrimination, ethics, fairness"


In [6]:
# Summary 2: pivot table showing category counts across companies

category_pivot = (
    company_category_summary
    .pivot_table(
        index=["company_id", "company_name"],
        columns="category",
        values="mention_count",
        fill_value=0
    )
    .reset_index()
)

category_pivot.to_csv(
    evidence_folder / "company_category_keyword_pivot.csv",
    index=False
)

category_pivot

category,company_id,company_name,AI and related technologies,Data and analytics,"Fairness, bias and ethics",Governance and accountability,Human oversight,Risk and regulation,Transparency and explainability
0,001,Barclays plc,231.0,10.0,20.0,927.0,1.0,358.0,41.0
1,002,HSBC Holdings plc,91.0,5.0,17.0,805.0,0.0,294.0,15.0
2,003,Lloyds Banking Group plc,76.0,4.0,10.0,303.0,0.0,200.0,16.0
3,004,NatWest Group plc,198.0,5.0,23.0,891.0,1.0,297.0,11.0
4,005,Standard Chartered plc,86.0,6.0,34.0,378.0,0.0,183.0,13.0
5,006,Aviva plc,90.0,2.0,16.0,568.0,0.0,89.0,5.0
6,007,Legal & General Group plc,45.0,2.0,10.0,263.0,0.0,87.0,23.0
7,008,M&G plc,20.0,0.0,5.0,552.0,0.0,134.0,13.0
8,009,Admiral Group plc,90.0,6.0,14.0,642.0,0.0,124.0,27.0
9,010,AJ Bell plc,18.0,0.0,5.0,346.0,0.0,108.0,8.0


In [7]:
# Summary 3: narrower AI-transparency-focused keyword set

ai_transparency_keywords = [
    "artificial intelligence",
    "AI",
    "machine learning",
    "algorithm",
    "automation",
    "generative AI",
    "large language model",
    "LLM",
    "responsible AI",
    "ethical AI",
    "AI ethics",
    "explainability",
    "explainable AI",
    "interpretability",
    "human oversight",
    "human review",
    "human-in-the-loop",
    "automated decision-making",
    "bias",
    "fairness"
]

ai_transparency_mentions = keyword_mentions[
    keyword_mentions["keyword"].isin(ai_transparency_keywords)
].copy()

ai_transparency_summary = (
    ai_transparency_mentions
    .groupby(["company_id", "company_name", "keyword"])
    .agg(
        mention_count=("keyword", "count"),
        unique_pages=("page_number", "nunique")
    )
    .reset_index()
    .sort_values(
        by=["company_id", "mention_count"],
        ascending=[True, False]
    )
)

ai_transparency_mentions.to_csv(
    evidence_folder / "ai_transparency_mentions_detailed.csv",
    index=False
)

ai_transparency_summary.to_csv(
    evidence_folder / "ai_transparency_keyword_summary.csv",
    index=False
)

print(f"AI-transparency-focused mentions found: {len(ai_transparency_mentions)}")

ai_transparency_summary.head(40)

AI-transparency-focused mentions found: 1448


,company_id,company_name,keyword,mention_count,unique_pages
0,001,Barclays plc,AI,195,47
1,001,Barclays plc,artificial intelligence,20,14
3,001,Barclays plc,bias,8,6
2,001,Barclays plc,automation,7,7
5,001,Barclays plc,fairness,7,7
8,001,Barclays plc,machine learning,7,6
6,001,Barclays plc,generative AI,2,2
4,001,Barclays plc,ethical AI,1,1
7,001,Barclays plc,human oversight,1,1
9,001,Barclays plc,responsible AI,1,1


In [8]:
# Company-level AI transparency summary

company_ai_summary = (
    ai_transparency_mentions
    .groupby(["company_id", "company_name"])
    .agg(
        total_ai_transparency_mentions=("keyword", "count"),
        unique_pages_with_mentions=("page_number", "nunique"),
        unique_keywords_found=("keyword", "nunique"),
        keywords_found=("keyword", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
    .sort_values(
        by="total_ai_transparency_mentions",
        ascending=False
    )
)

company_ai_summary.to_csv(
    evidence_folder / "company_ai_transparency_summary.csv",
    index=False
)

company_ai_summary

,company_id,company_name,total_ai_transparency_mentions,unique_pages_with_mentions,unique_keywords_found,keywords_found
0,001,Barclays plc,249,60,10,"AI, artificial intelligence, automation, bias,..."
3,004,NatWest Group plc,210,61,10,"AI, AI ethics, artificial intelligence, automa..."
18,019,London Stock Exchange Group plc,119,34,8,"AI, LLM, artificial intelligence, automation, ..."
15,016,Funding Circle Holdings plc,104,30,8,"AI, artificial intelligence, automation, bias,..."
8,009,Admiral Group plc,103,39,8,"AI, artificial intelligence, automation, bias,..."
1,002,HSBC Holdings plc,100,39,10,"AI, AI ethics, artificial intelligence, automa..."
4,005,Standard Chartered plc,94,33,7,"AI, artificial intelligence, automation, bias,..."
5,006,Aviva plc,91,33,5,"AI, artificial intelligence, automation, fairn..."
2,003,Lloyds Banking Group plc,79,35,8,"AI, AI ethics, artificial intelligence, automa..."
19,020,Experian plc,74,35,7,"AI, artificial intelligence, automation, bias,..."


In [9]:
# Pivot table: companies by AI-transparency keywords

ai_keyword_pivot = (
    ai_transparency_summary
    .pivot_table(
        index=["company_id", "company_name"],
        columns="keyword",
        values="mention_count",
        fill_value=0
    )
    .reset_index()
)

ai_keyword_pivot.to_csv(
    evidence_folder / "ai_transparency_keyword_pivot.csv",
    index=False
)

ai_keyword_pivot

keyword,company_id,company_name,AI,AI ethics,LLM,artificial intelligence,automated decision-making,automation,bias,ethical AI,fairness,generative AI,human oversight,human-in-the-loop,large language model,machine learning,responsible AI
0,001,Barclays plc,195.0,0.0,0.0,20.0,0.0,7.0,8.0,1.0,7.0,2.0,1.0,0.0,0.0,7.0,1.0
1,002,HSBC Holdings plc,73.0,1.0,0.0,5.0,0.0,4.0,4.0,0.0,3.0,4.0,0.0,0.0,1.0,4.0,1.0
2,003,Lloyds Banking Group plc,58.0,1.0,0.0,9.0,0.0,3.0,1.0,1.0,0.0,5.0,0.0,0.0,0.0,1.0,0.0
3,004,NatWest Group plc,144.0,1.0,0.0,27.0,1.0,15.0,4.0,0.0,2.0,10.0,0.0,0.0,0.0,2.0,4.0
4,005,Standard Chartered plc,79.0,0.0,0.0,3.0,0.0,3.0,2.0,0.0,3.0,1.0,0.0,0.0,0.0,0.0,3.0
5,006,Aviva plc,65.0,0.0,0.0,17.0,0.0,6.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0
6,007,Legal & General Group plc,43.0,0.0,0.0,0.0,0.0,1.0,5.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
7,008,M&G plc,13.0,0.0,0.0,4.0,0.0,3.0,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
8,009,Admiral Group plc,71.0,0.0,0.0,8.0,0.0,3.0,1.0,0.0,5.0,3.0,0.0,0.0,0.0,5.0,7.0
9,010,AJ Bell plc,5.0,0.0,0.0,2.0,0.0,8.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Created a smaller evidence sample for manual review

manual_review_sample = (
    ai_transparency_mentions
    .sort_values(
        by=["company_id", "page_number", "keyword"]
    )
    .groupby(["company_id", "company_name", "keyword"])
    .head(3)
    .reset_index(drop=True)
)

manual_review_sample.to_csv(
    evidence_folder / "manual_review_sample_ai_mentions.csv",
    index=False
)

manual_review_sample.head(50)

,company_id,company_name,report_year,source_file,category,keyword,search_variant,page_number,matched_text,context
0,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,"Fairness, bias and ethics",fairness,fairness,2,fairness,"d opportunity in courage, transparency and hum..."
1,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,AI,AI,6,AI,rough 2025 with rapid We are guided by our Val...
2,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,AI,AI,6,AI,bringing both opportunity and risk. Barclays i...
3,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,AI,AI,7,AI,to deepen our grasp of the opportunities and c...
4,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,automation,automation,22,automation,"control resilience, including by increasing ou..."
5,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,generative AI,generative AI,22,generative AI,"folios, and maintained their strong underlying..."
6,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,42,artificial intelligence,class customer experience; and delivering best...
7,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,automation,automation,43,automation,ith colleagues working on The Board has contin...
8,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,62,artificial intelligence,ds (including business or operations; the use ...
9,001,Barclays plc,2025,001_barclays_plc_annual_report_2025.txt,AI and related technologies,artificial intelligence,artificial intelligence,109,artificial intelligence,ustainability reporting standards (including b...
